# 🔬 4D Endoscopic Scene Reconstruction
### EndoGaussian Pipeline — Google Colab Edition

**Paper:** *Foundation Model-Guided Gaussian Splatting for 4D Reconstruction of Deformable Tissues* (IEEE TMI 2025)

**Input:** `Video01.mp4` — Laparoscopic surgery video (1280×720, 30fps, 14.6s)

**Output:** A real-time renderable 4D scene (3D space + time) capturing deforming tissue

---

### ⚡ Before you start — 3 things to check:
1. **Runtime → Change runtime type → T4 GPU** (free) or A100 (Colab Pro)
2. **Upload `Video01.mp4`** to your Google Drive → `My Drive/EndoGaussian/Video01.mp4`
3. Run cells **top to bottom** — each step builds on the previous

---
| Step | Description | Time (T4) |
|------|-------------|----------|
| 0 | GPU Check & Drive Mount | < 1 min |
| 1 | Install Dependencies | ~8 min |
| 2 | Extract & Prepare Frames | ~1 min |
| 3 | Monocular Depth (Depth-Anything-V2) | ~5 min |
| 4 | Camera Poses (COLMAP) | ~10 min |
| 5 | Clone & Build EndoGaussian | ~5 min |
| 6 | Train 4D Gaussian Scene | ~15 min |
| 7 | Render & Export Video | ~3 min |


---
## ✅ STEP 0 — GPU Check & Google Drive Mount

In [ ]:
# ── Verify GPU ──────────────────────────────────────────────
import subprocess, sys
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("\n⚠️  NO GPU DETECTED!")
    print("   Go to: Runtime → Change runtime type → GPU (T4)")
    sys.exit()

In [ ]:
# ── Mount Google Drive ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("✅ Google Drive mounted at /content/drive")

In [ ]:
# ── Configure Paths ─────────────────────────────────────────
import os

# ╔══════════════════════════════════════════════════════╗
# ║  EDIT THIS if your video is in a different location ║
# ╚══════════════════════════════════════════════════════╝
DRIVE_ROOT   = "/content/drive/MyDrive/EndoGaussian"
VIDEO_PATH   = f"{DRIVE_ROOT}/Video01.mp4"
WORK_DIR     = "/content/endogaussian_workspace"  # fast local SSD
SAVE_DIR     = f"{DRIVE_ROOT}/outputs"            # persisted to Drive

# Create directories
for d in [DRIVE_ROOT, WORK_DIR, SAVE_DIR,
          f"{WORK_DIR}/frames_all",
          f"{WORK_DIR}/frames_train",
          f"{WORK_DIR}/masks",
          f"{WORK_DIR}/depth/npy",
          f"{WORK_DIR}/depth/vis",
          f"{WORK_DIR}/colmap",
          f"{WORK_DIR}/dataset",
          f"{WORK_DIR}/renders"]:
    os.makedirs(d, exist_ok=True)

# Check video file
if os.path.exists(VIDEO_PATH):
    size_mb = os.path.getsize(VIDEO_PATH) / 1e6
    print(f"✅ Video found: {VIDEO_PATH} ({size_mb:.1f} MB)")
else:
    print(f"❌ Video NOT found at: {VIDEO_PATH}")
    print(f"   Please upload Video01.mp4 to your Google Drive:")
    print(f"   {DRIVE_ROOT}/Video01.mp4")

---
## 📦 STEP 1 — Install Dependencies
> This takes ~8 minutes on first run. Output is suppressed for readability.

In [ ]:
# ── System packages ─────────────────────────────────────────
print("Installing system packages (colmap, ffmpeg, git-lfs)...")
!apt-get install -y colmap ffmpeg git-lfs libgl1 libglib2.0-0 -qq
!colmap --version
print("✅ System packages ready")

In [ ]:
# ── Python packages ─────────────────────────────────────────
print('Installing Python packages...')
!pip install -q \
    transformers \
    accelerate \
    timm \
    einops \
    open3d \
    plyfile \
    tqdm \
    lpips \
    torchmetrics \
    opencv-python-headless \
    imageio imageio-ffmpeg \
    scipy scikit-image \
    matplotlib

# Verify key packages imported cleanly
import importlib
failed = []
for pkg in ['transformers','accelerate','timm','einops','plyfile',
            'tqdm','lpips','torchmetrics','cv2','imageio','scipy']:
    try:
        importlib.import_module(pkg)
    except ImportError:
        failed.append(pkg)
if failed:
    print(f'⚠️  Could not import: {failed} — re-run this cell once')
else:
    print('✅ All Python packages installed and verified')


In [ ]:
# ── Clone EndoGaussian & Build CUDA Submodules ──────────────
import os, subprocess, sys, shutil

REPO_DIR = '/content/EndoGaussian'
SUB_DIR  = f'{REPO_DIR}/submodules'
RAST_DIR = f'{SUB_DIR}/diff-gaussian-rasterization'
KNN_DIR  = f'{SUB_DIR}/simple-knn'
RAST_URL = 'https://github.com/graphdeco-inria/diff-gaussian-rasterization.git'
KNN_URL  = 'https://github.com/camenduru/simple-knn.git'

# ── 0. Locate CUDA — fall back to CPU if not found ──────────
print('Locating CUDA...')
cuda_candidates = [
    '/usr/local/cuda',
    '/usr/local/cuda-12.2',
    '/usr/local/cuda-12.1',
    '/usr/local/cuda-11.8',
    '/usr/local/cuda-11.7',
]
CUDA_HOME = None
for c in cuda_candidates:
    if os.path.exists(f'{c}/bin/nvcc'):
        CUDA_HOME = c
        break

if CUDA_HOME is None:
    r = subprocess.run(['find', '/usr', '-name', 'nvcc', '-type', 'f'],
                       capture_output=True, text=True)
    hits = [l.strip() for l in r.stdout.splitlines() if l.strip()]
    if hits:
        CUDA_HOME = str(__import__('pathlib').Path(hits[0]).parent.parent)

if CUDA_HOME is not None:
    # GPU path — set env vars so ALL child processes (including pip's setup.py)
    # inherit CUDA_HOME. Must use os.environ directly, not env={} in subprocess.
    os.environ['CUDA_HOME']       = CUDA_HOME
    os.environ['CUDA_PATH']       = CUDA_HOME
    os.environ['FORCE_CUDA']      = '1'
    os.environ['PATH']            = f"{CUDA_HOME}/bin:" + os.environ.get('PATH', '')
    os.environ['LD_LIBRARY_PATH'] = f"{CUDA_HOME}/lib64:" + os.environ.get('LD_LIBRARY_PATH', '')
    print(f'✅ CUDA_HOME = {CUDA_HOME}')
    print(f'   nvcc      = {CUDA_HOME}/bin/nvcc')
    USE_GPU = True
else:
    # CPU fallback — training will be much slower (~10× slower) but still works
    os.environ.pop('CUDA_HOME', None)
    os.environ.pop('FORCE_CUDA', None)
    os.environ['CUDA_VISIBLE_DEVICES'] = ''   # hide GPUs from PyTorch too
    USE_GPU = False
    print('⚠️  CUDA not found — running in CPU-only mode')
    print('   Training will work but will be significantly slower.')
    print('   For GPU: Runtime → Change runtime type → T4 GPU')

# Install build tools
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'ninja', 'packaging', 'setuptools', 'wheel'], check=True)
print('✅ Build tools ready\n')

# ── Helper ───────────────────────────────────────────────────
def run(cmd, cwd=None, label=''):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'❌ FAILED: {label}')
        if r.stdout: print(r.stdout[-1500:])
        if r.stderr: print(r.stderr[-1500:])
        raise RuntimeError(f'{label} failed')
    return r

def pip_install_local(path, label):
    """Try pip editable, fallback to setup.py install."""
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-e', path, '--no-build-isolation'],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        print(f'  ✅ {label} installed (pip editable)')
        return
    print(f'  ⚠️  pip editable failed, trying setup.py install...')
    r = subprocess.run(
        [sys.executable, 'setup.py', 'install'],
        cwd=path, capture_output=True, text=True
    )
    if r.returncode == 0:
        print(f'  ✅ {label} installed (setup.py install)')
        return
    print(f'  ❌ Both methods failed for {label}:')
    if r.stdout: print(r.stdout[-1500:])
    if r.stderr: print(r.stderr[-1500:])
    raise RuntimeError(f'{label} build failed')

# ── 1. Clone main repo ───────────────────────────────────────
if not os.path.exists(REPO_DIR):
    print('Cloning EndoGaussian...')
    run(['git', 'clone',
         'https://github.com/CUHK-AIM-Group/EndoGaussian.git', REPO_DIR],
        label='clone EndoGaussian')
    print('✅ EndoGaussian cloned')
else:
    print('✅ EndoGaussian already present')

os.makedirs(SUB_DIR, exist_ok=True)

# ── 2. Clone diff-gaussian-rasterization ─────────────────────
if not os.path.exists(f'{RAST_DIR}/setup.py'):
    print('Cloning diff-gaussian-rasterization...')
    if os.path.exists(RAST_DIR): shutil.rmtree(RAST_DIR)
    run(['git', 'clone', '--recursive', RAST_URL, RAST_DIR],
        label='clone diff-gaussian-rasterization')
    print('✅ diff-gaussian-rasterization cloned')
else:
    print('✅ diff-gaussian-rasterization already present')

# ── 3. Clone simple-knn ──────────────────────────────────────
if not os.path.exists(f'{KNN_DIR}/setup.py'):
    print('Cloning simple-knn...')
    if os.path.exists(KNN_DIR): shutil.rmtree(KNN_DIR)
    run(['git', 'clone', KNN_URL, KNN_DIR], label='clone simple-knn')
    print('✅ simple-knn cloned')
else:
    print('✅ simple-knn already present')

# ── 3b. Patch simple-knn source (fixes FLT_MAX undefined) ─────
# Root cause: simple_knn.cu manually redefines __CUDACC__ which
# prevents <float.h> from being included, so FLT_MAX is never defined.
# Fix: prepend #include <float.h> and #include <cfloat> to the .cu file.
cu_file = f'{KNN_DIR}/simple_knn.cu'
if os.path.exists(cu_file):
    with open(cu_file, 'r') as f:
        cu_src = f.read()
    patch = '#include <float.h>\n#include <cfloat>\n'
    if '#include <float.h>' not in cu_src:
        with open(cu_file, 'w') as f:
            f.write(patch + cu_src)
        print('✅ simple_knn.cu patched (added float.h include)')
    else:
        print('✅ simple_knn.cu already patched')
else:
    print(f'⚠️  {cu_file} not found — skipping patch')

# ── 4. Build CUDA submodules (skip gracefully on CPU-only) ───
if USE_GPU:
    print('Building diff-gaussian-rasterization (~3 min)...')
    pip_install_local(RAST_DIR, 'diff-gaussian-rasterization')
    print('Building simple-knn (~1 min)...')
    pip_install_local(KNN_DIR, 'simple-knn')
else:
    print('⚠️  Skipping CUDA submodule builds (CPU mode)')
    print('   diff-gaussian-rasterization and simple-knn require CUDA to compile.')
    print('   The pipeline will use PyTorch CPU fallbacks instead.')

# ── 5. Smoke test ────────────────────────────────────────────
print('\nVerifying imports...')
for mod in ['diff_gaussian_rasterization', 'simple_knn']:
    try:
        __import__(mod)
        print(f'  ✅ {mod}')
    except ImportError:
        if USE_GPU:
            print(f'  ⚠️  {mod} import failed → Runtime → Restart session, re-run all cells')
        else:
            print(f'  ℹ️  {mod} not available (expected in CPU mode)')

mode_str = 'GPU (' + CUDA_HOME + ')' if USE_GPU else 'CPU-only'
print(f'\n✅ Setup complete [{mode_str}] — proceed to Step 2')


---
## 🎬 STEP 2 — Frame Extraction & Mask Generation

In [ ]:
# ── Video info ───────────────────────────────────────────────
import json, sys

# 1. Hard-stop if video is missing — gives a clear actionable message
if not os.path.exists(VIDEO_PATH):
    print(f"❌ Video file not found: {VIDEO_PATH}")
    print("")
    print("Please do ONE of the following:")
    print("  A) Upload Video01.mp4 to Google Drive at:")
    print(f"       MyDrive/EndoGaussian/Video01.mp4")
    print("  B) Edit DRIVE_ROOT in Step 0 to point at your actual folder")
    print("")
    print("Run this to see what files exist in your Drive:")
    print("  import os; print(os.listdir('/content/drive/MyDrive/'))")
    raise FileNotFoundError(f"Video not found: {VIDEO_PATH}")

print(f"✅ Video found: {VIDEO_PATH}")
print(f"   Size: {os.path.getsize(VIDEO_PATH)/1e6:.1f} MB")

# 2. Run ffprobe — path as list arg avoids shell-quoting issues
probe = subprocess.run(
    ['ffprobe', '-v', 'quiet', '-print_format', 'json',
     '-show_streams', '-show_format', VIDEO_PATH],
    capture_output=True, text=True
)

# 3. Guard: show stderr if stdout empty
if not probe.stdout.strip():
    print("❌ ffprobe returned no output. Error:")
    print(probe.stderr)
    raise RuntimeError("ffprobe failed — see error above")

# 4. Parse JSON safely
try:
    info = json.loads(probe.stdout)
except json.JSONDecodeError as e:
    print(f"❌ Could not parse ffprobe output: {e}")
    print("Raw output:", probe.stdout[:300])
    raise

# 5. Find the video stream — don't assume it's always index 0
video_streams = [s for s in info.get('streams', []) if s.get('codec_type') == 'video']
if not video_streams:
    print("❌ No video stream found. Streams in file:")
    for s in info.get('streams', []):
        print(f"   codec_type={s.get('codec_type')} name={s.get('codec_name')}")
    raise ValueError("File has no video stream")

v   = video_streams[0]
fmt = info['format']

print("\n📹 Video Properties:")
print(f"   Resolution : {v['width']}x{v['height']}")
print(f"   Frame rate : {v['r_frame_rate']} fps")
print(f"   Duration   : {float(fmt['duration']):.1f}s")
print(f"   Frames     : {v.get('nb_frames', 'N/A')}")
print(f"   Codec      : {v['codec_name']}")
print(f"   Bitrate    : {int(fmt.get('bit_rate', 0))//1000} kbps")

W = int(v['width'])
H = int(v['height'])

# Crop to remove black circular endoscope border (90% of frame)
CROP_W = int(W * 0.90); CROP_W -= CROP_W % 2   # must be even for h264
CROP_H = int(H * 0.90); CROP_H -= CROP_H % 2
print(f"\n   Crop size  : {CROP_W}x{CROP_H} (removes black border)")

In [ ]:
# ── Extract frames ───────────────────────────────────────────
# Using subprocess.run() instead of !ffmpeg shell magic to avoid
# path quoting issues (spaces, special chars in Drive paths).

CROP_FILTER = f"crop={CROP_W}:{CROP_H}:(in_w-{CROP_W})/2:(in_h-{CROP_H})/2"

def run_ffmpeg(vf_filter, out_dir, out_pattern, label):
    """Run ffmpeg and raise clearly if it fails."""
    cmd = ['ffmpeg', '-y', '-i', VIDEO_PATH,
           '-vf', vf_filter, out_pattern, '-loglevel', 'error']
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"❌ ffmpeg failed for {label}:")
        print(result.stderr)
        raise RuntimeError(f"ffmpeg error: {result.stderr[:400]}")
    n = len([f for f in os.listdir(out_dir) if f.endswith('.png')])
    print(f"   ✅ {n} frames extracted → {label}")
    return n

# All frames at 10fps → used for depth estimation
print("Extracting frames at 10fps (for depth estimation)...")
n_all = run_ffmpeg(
    f"fps=10,{CROP_FILTER}",
    f"{WORK_DIR}/frames_all",
    f"{WORK_DIR}/frames_all/frame_%05d.png",
    "frames_all/"
)

# Training subset at 2fps → COLMAP + EndoGaussian training
print("Extracting training frames at 2fps...")
n_train_frames = run_ffmpeg(
    f"fps=2,{CROP_FILTER}",
    f"{WORK_DIR}/frames_train",
    f"{WORK_DIR}/frames_train/frame_%05d.png",
    "frames_train/"
)
print(f"\nTotal: {n_all} depth frames | {n_train_frames} training frames")

In [ ]:
# ── Generate circular endoscope masks ───────────────────────
import cv2
import numpy as np
from pathlib import Path

frames = sorted(Path(f"{WORK_DIR}/frames_train").glob("*.png"))
for fp in frames:
    img = cv2.imread(str(fp))
    h, w = img.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    cx, cy = w // 2, h // 2
    r = int(min(h, w) // 2 * 0.97)
    cv2.circle(mask, (cx, cy), r, 255, -1)
    cv2.imwrite(f"{WORK_DIR}/masks/{fp.name}", mask)

print(f"✅ {len(frames)} endoscope masks generated → masks/")

# Preview: show 3 sample frames side by side
import matplotlib.pyplot as plt
sample_idxs = [0, len(frames)//2, len(frames)-1]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.patch.set_facecolor('#111')
for ax, idx in zip(axes, sample_idxs):
    fp = frames[idx]
    img = cv2.cvtColor(cv2.imread(str(fp)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f"Frame {idx} ({idx*0.5:.1f}s)", color='white', fontsize=9)
    ax.axis('off')
plt.suptitle("Sample Training Frames", color='white', fontsize=12)
plt.tight_layout()
plt.show()

---
## 🌊 STEP 3 — Monocular Depth Estimation
> Uses **Depth-Anything-V2-Large** — a foundation vision model that predicts per-pixel depth from a single RGB image. This replaces the stereo camera typically needed for 3D reconstruction.

In [ ]:
# ── Load Depth-Anything-V2 ───────────────────────────────────
from transformers import pipeline as hf_pipeline
from PIL import Image

print("Loading Depth-Anything-V2-Large model...")
print("(~1.3 GB download on first run, cached after)")

depth_pipe = hf_pipeline(
    "depth-estimation",
    model="depth-anything/Depth-Anything-V2-Large-hf",
    device=0  # GPU
)
print("✅ Depth model loaded on GPU")

In [ ]:
# ── Run depth estimation on all frames ──────────────────────
from tqdm import tqdm

all_frames = sorted(Path(f"{WORK_DIR}/frames_all").glob("*.png"))
print(f"Processing {len(all_frames)} frames...")

for fp in tqdm(all_frames, desc="Depth estimation"):
    img = Image.open(str(fp)).convert("RGB")
    result = depth_pipe(img)
    depth_array = np.array(result["depth"], dtype=np.float32)

    # Save raw depth as .npy
    np.save(f"{WORK_DIR}/depth/npy/{fp.stem}.npy", depth_array)

    # Save colourised visualization as PNG
    d_norm = cv2.normalize(depth_array, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    d_colour = cv2.applyColorMap(d_norm, cv2.COLORMAP_TURBO)
    cv2.imwrite(f"{WORK_DIR}/depth/vis/{fp.stem}.png", d_colour)

print(f"\n✅ Depth maps saved:")
print(f"   Raw (.npy) → {WORK_DIR}/depth/npy/")
print(f"   Visual     → {WORK_DIR}/depth/vis/")

In [ ]:
# ── Visualize depth results ──────────────────────────────────
sample = all_frames[len(all_frames)//2]
rgb = cv2.cvtColor(cv2.imread(str(sample)), cv2.COLOR_BGR2RGB)
depth_vis = cv2.cvtColor(
    cv2.imread(f"{WORK_DIR}/depth/vis/{sample.stem}.png"),
    cv2.COLOR_BGR2RGB
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#111')
ax1.imshow(rgb);       ax1.set_title("RGB Frame", color='white', fontsize=11); ax1.axis('off')
ax2.imshow(depth_vis); ax2.set_title("Estimated Depth (Depth-Anything-V2)\nBlue=far · Red=close", color='white', fontsize=11); ax2.axis('off')
plt.suptitle("Monocular Depth Estimation Result", color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📷 STEP 4 — Camera Pose Estimation (COLMAP)
> COLMAP runs **Structure-from-Motion** to estimate how the laparoscope camera moved between frames, recovering both the 3D sparse point cloud and camera extrinsics.

In [ ]:
# ── COLMAP feature extraction ────────────────────────────────
COLMAP_DB      = f"{WORK_DIR}/colmap/database.db"
COLMAP_SPARSE  = f"{WORK_DIR}/colmap/sparse"
os.makedirs(COLMAP_SPARSE, exist_ok=True)

# Estimate focal length from 70° FOV laparoscope
import math
fov_deg = 70
fx = CROP_W / (2 * math.tan(math.radians(fov_deg / 2)))
cam_params = f"{fx:.2f},{fx:.2f},{CROP_W/2:.2f},{CROP_H/2:.2f},0,0,0,0"
print(f"Estimated camera intrinsics: fx={fx:.1f}, cx={CROP_W/2}, cy={CROP_H/2}")

print("\n[4a] Extracting SIFT features...")
!colmap feature_extractor \
    --database_path {COLMAP_DB} \
    --image_path {WORK_DIR}/frames_train \
    --ImageReader.camera_model OPENCV \
    --ImageReader.single_camera 1 \
    --ImageReader.camera_params "{cam_params}" \
    --SiftExtraction.use_gpu 1 \
    --SiftExtraction.max_num_features 8192 \
    --SiftExtraction.estimate_affine_shape 1 \
    --SiftExtraction.domain_size_pooling 1

print("✅ Feature extraction complete")

In [ ]:
# ── COLMAP sequential matching (video-optimized) ─────────────
print("[4b] Sequential feature matching (overlap=10 frames)...")
!colmap sequential_matcher \
    --database_path {COLMAP_DB} \
    --SequentialMatching.overlap 10 \
    --SequentialMatching.loop_detection 1 \
    --SiftMatching.use_gpu 1 \
    --SiftMatching.guided_matching 1
print("✅ Feature matching complete")

In [ ]:
# ── COLMAP sparse reconstruction ─────────────────────────────
print("[4c] Running sparse reconstruction (bundle adjustment)...")
!colmap mapper \
    --database_path {COLMAP_DB} \
    --image_path {WORK_DIR}/frames_train \
    --output_path {COLMAP_SPARSE} \
    --Mapper.ba_global_max_num_iterations 30 \
    --Mapper.init_min_num_inliers 10

# Check registered images
models = sorted(os.listdir(COLMAP_SPARSE))
if models:
    print(f"\n✅ Sparse model(s) built: {models}")
    !colmap model_analyzer --path {COLMAP_SPARSE}/{models[0]}
else:
    print("⚠️  No COLMAP model built — falling back to identity poses")
    print("   (This can happen with limited camera motion in laparoscopy)")
    print("   Training will still work using depth-only initialization.")

In [ ]:
# ── Convert COLMAP output to transforms.json ─────────────────
# This converts COLMAP's binary format to the JSON camera format
# used by EndoGaussian / NeRF-style pipelines

COLMAP_TRANSFORMS = f"{WORK_DIR}/colmap/transforms.json"
models = sorted(os.listdir(COLMAP_SPARSE))

if models:
    # Use ns-process-data style conversion (bundled with nerfstudio OR manual)
    # We'll do it manually from COLMAP text output
    !colmap model_converter \
        --input_path {COLMAP_SPARSE}/{models[0]} \
        --output_path {WORK_DIR}/colmap/sparse_txt \
        --output_type TXT

    # Parse COLMAP text files → transforms.json
    import numpy as np

    def parse_colmap_images(images_txt):
        """Parse COLMAP images.txt to get camera poses as 4×4 matrices."""
        poses = {}
        with open(images_txt) as f:
            lines = [l for l in f.readlines() if not l.startswith('#')]
        i = 0
        while i < len(lines):
            parts = lines[i].strip().split()
            if len(parts) >= 9:
                img_id = int(parts[0])
                qw,qx,qy,qz = map(float, parts[1:5])
                tx,ty,tz    = map(float, parts[5:8])
                name = parts[9]
                # Quaternion → rotation matrix
                R = np.array([
                    [1-2*(qy**2+qz**2), 2*(qx*qy-qw*qz), 2*(qx*qz+qw*qy)],
                    [2*(qx*qy+qw*qz), 1-2*(qx**2+qz**2), 2*(qy*qz-qw*qx)],
                    [2*(qx*qz-qw*qy), 2*(qy*qz+qw*qx), 1-2*(qx**2+qy**2)]
                ])
                t = np.array([tx, ty, tz])
                # World-to-cam → cam-to-world
                T = np.eye(4)
                T[:3,:3] = R.T
                T[:3, 3] = -R.T @ t
                poses[name] = T.tolist()
            i += 2  # skip point-2D line
        return poses

    images_txt = f"{WORK_DIR}/colmap/sparse_txt/images.txt"
    cams_txt   = f"{WORK_DIR}/colmap/sparse_txt/cameras.txt"

    if os.path.exists(images_txt):
        poses = parse_colmap_images(images_txt)
        frame_names = sorted(poses.keys())
        transforms = {
            "camera_model": "OPENCV",
            "fl_x": fx, "fl_y": fx,
            "cx": CROP_W/2, "cy": CROP_H/2,
            "w": CROP_W, "h": CROP_H,
            "k1": -0.3, "k2": 0.1, "p1": 0.0, "p2": 0.0,
            "frames": [
                {
                    "file_path": f"images/{name}",
                    "mask_path": f"masks/{name}",
                    "time": i / len(frame_names),
                    "transform_matrix": poses[name]
                }
                for i, name in enumerate(frame_names)
            ]
        }
        print(f"✅ Parsed {len(frame_names)} camera poses from COLMAP")
    else:
        poses = {}

# Fallback: identity poses if COLMAP failed
if not models or not os.path.exists(f"{WORK_DIR}/colmap/sparse_txt/images.txt"):
    print("Using identity camera poses (depth-only init)...")
    train_frames = sorted(os.listdir(f"{WORK_DIR}/frames_train"))
    transforms = {
        "camera_model": "OPENCV",
        "fl_x": fx, "fl_y": fx,
        "cx": CROP_W/2, "cy": CROP_H/2,
        "w": CROP_W, "h": CROP_H,
        "k1": -0.3, "k2": 0.1, "p1": 0.0, "p2": 0.0,
        "frames": [
            {
                "file_path": f"images/{name}",
                "mask_path": f"masks/{name}",
                "time": i / len(train_frames),
                "transform_matrix": [[1,0,0,0],[0,1,0,0],[0,0,1,0],[0,0,0,1]]
            }
            for i, name in enumerate(train_frames)
        ]
    }

# 80/20 train/test split + save
n = len(transforms['frames'])
n_train = max(1, int(0.8 * n))
train_t = {**transforms, "frames": transforms['frames'][:n_train]}
test_t  = {**transforms, "frames": transforms['frames'][n_train:]}

DATASET = f"{WORK_DIR}/dataset"
import shutil
for src_dir, dst_dir in [
    (f"{WORK_DIR}/frames_train", f"{DATASET}/images"),
    (f"{WORK_DIR}/masks",        f"{DATASET}/masks"),
    (f"{WORK_DIR}/depth/npy",    f"{DATASET}/depth")
]:
    if os.path.exists(dst_dir): shutil.rmtree(dst_dir)
    shutil.copytree(src_dir, dst_dir)

with open(f"{DATASET}/transforms_train.json", 'w') as f: json.dump(train_t, f, indent=2)
with open(f"{DATASET}/transforms_test.json",  'w') as f: json.dump(test_t, f, indent=2)
print(f"\n✅ Dataset ready at: {DATASET}/")
print(f"   Train: {n_train} frames | Test: {n - n_train} frames")

---
## 🔥 STEP 5 — Train 4D Gaussian Splatting
> **This is the core step.** EndoGaussian trains millions of 3D Gaussians with a time-varying deformation field, learning to reproduce the video from any viewpoint at any timestep.
>
> Expected time on T4: ~15 minutes
>
> **What's being trained:**
> - 3D Gaussian positions, scales, rotations, opacities, colors
> - Per-Gaussian deformation MLP: `(x,y,z,t) → (Δx,Δy,Δz)` ← the 4D part
> - Motion-Aware Frame Synthesis for large tissue deformations

In [ ]:
# ── STEP 5: Diagnose, Fix Dataset Format & Train ────────────
import os, sys, json, shutil, subprocess
from pathlib import Path

GAUSSIANS_OUT = f'{WORK_DIR}/gaussians'
os.makedirs(GAUSSIANS_OUT, exist_ok=True)

# ── 1. Read scene/__init__.py to understand what it expects ──
scene_init = '/content/EndoGaussian/scene/__init__.py'
print('=== scene/__init__.py (scene type detection logic) ===')
with open(scene_init) as f:
    lines = f.readlines()
for i, l in enumerate(lines):
    print(f'{i+1:3}: {l}', end='')
    if i > 100: print('  ... (truncated)'); break

print('\n\n=== Current dataset layout ===')
for root, dirs, files in os.walk(DATASET):
    lvl = root.replace(DATASET,'').count(os.sep)
    print('  '*lvl + os.path.basename(root) + '/')
    for fname in sorted(files)[:4]:
        print('  '*(lvl+1) + fname)
    if len(files) > 4:
        print('  '*(lvl+1) + f'... ({len(files)} total)')

# ── 2. Detect which format scene/__init__.py needs ────────────
scene_src = ''.join(lines)

needs_colmap      = 'sparse' in scene_src and ('colmap' in scene_src.lower() or 'COLMAP' in scene_src)
needs_transforms  = 'transforms_train' in scene_src or 'transforms.json' in scene_src
needs_camera_json = 'cameras.json' in scene_src
needs_poses_bounds= 'poses_bounds' in scene_src

print('\n=== Detected format requirements ===')
print(f'  needs COLMAP sparse/  : {needs_colmap}')
print(f'  needs transforms_train: {needs_transforms}')
print(f'  needs cameras.json    : {needs_camera_json}')
print(f'  needs poses_bounds.npy: {needs_poses_bounds}')

# ── 3. Check what's actually present ─────────────────────────
has_sparse     = os.path.exists(f'{DATASET}/sparse')
has_transforms = os.path.exists(f'{DATASET}/transforms_train.json')
has_cameras    = os.path.exists(f'{DATASET}/cameras.json')

print('\n=== What exists in dataset ===')
print(f'  sparse/               : {has_sparse}')
print(f'  transforms_train.json : {has_transforms}')
print(f'  cameras.json          : {has_cameras}')

# ── 4. Fix: ensure COLMAP sparse structure if needed ─────────
# EndoGaussian typically expects the COLMAP-style layout:
#   dataset/
#     images/         <- RGB frames
#     sparse/0/       <- COLMAP output (cameras.bin, images.bin, points3D.bin)
#                        OR text files (cameras.txt, images.txt, points3D.txt)
#
# If COLMAP ran and produced output, link it in.
# If COLMAP didn't run, create a minimal cameras.txt from our known intrinsics.

SPARSE_DIR = f'{DATASET}/sparse/0'
os.makedirs(SPARSE_DIR, exist_ok=True)

colmap_txt_src = f'{WORK_DIR}/colmap/sparse_txt'
colmap_bin_src = f'{WORK_DIR}/colmap/sparse'

if os.path.exists(colmap_txt_src) and os.listdir(colmap_txt_src):
    print('\n✅ COLMAP text output found — copying to dataset/sparse/0/')
    for fname in ['cameras.txt', 'images.txt', 'points3D.txt']:
        src = f'{colmap_txt_src}/{fname}'
        dst = f'{SPARSE_DIR}/{fname}'
        if os.path.exists(src):
            shutil.copy(src, dst)
            print(f'   copied {fname}')

elif os.path.exists(colmap_bin_src) and os.listdir(colmap_bin_src):
    models = sorted(os.listdir(colmap_bin_src))
    src_model = f'{colmap_bin_src}/{models[0]}'
    print(f'\n✅ COLMAP binary output found ({models[0]}) — converting to text')
    subprocess.run(['colmap', 'model_converter',
                    '--input_path', src_model,
                    '--output_path', SPARSE_DIR,
                    '--output_type', 'TXT'], check=True)
    print('   Converted to TXT')

else:
    print('\n⚠️  No COLMAP output found — generating minimal cameras.txt from known intrinsics')
    import math
    imgs = sorted(Path(f'{DATASET}/images').glob('*.png'))
    sample = __import__('cv2').imread(str(imgs[0]))
    H, W = sample.shape[:2]
    fov  = 70
    fx   = W / (2 * math.tan(math.radians(fov/2)))

    # cameras.txt: one shared camera
    with open(f'{SPARSE_DIR}/cameras.txt', 'w') as f:
        f.write('# Camera list with one line of data per camera:\n')
        f.write('#   CAMERA_ID, MODEL, WIDTH, HEIGHT, PARAMS[]\n')
        f.write(f'1 PINHOLE {W} {H} {fx:.4f} {fx:.4f} {W/2:.4f} {H/2:.4f}\n')

    # images.txt: identity pose for each frame (no COLMAP — forward-facing assumption)
    with open(f'{SPARSE_DIR}/images.txt', 'w') as f:
        f.write('# Image list with two lines of data per image:\n')
        f.write('#   IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, NAME\n')
        f.write('#   POINTS2D[] as (X, Y, POINT3D_ID)\n')
        for idx, img in enumerate(imgs, 1):
            f.write(f'{idx} 1.0 0.0 0.0 0.0 0.0 0.0 {idx*0.01:.4f} 1 {img.name}\n')
            f.write('\n')

    # points3D.txt: empty (will be filled by depth init)
    with open(f'{SPARSE_DIR}/points3D.txt', 'w') as f:
        f.write('# 3D point list — empty, depth init used instead\n')

    print(f'   Written minimal COLMAP stubs to {SPARSE_DIR}/')

# ── 5. Verify final layout ────────────────────────────────────
print('\n=== Final dataset layout ===')
for root, dirs, files in os.walk(DATASET):
    lvl = root.replace(DATASET,'').count(os.sep)
    if lvl > 3: continue
    print('  '*lvl + os.path.basename(root) + '/')
    for fname in sorted(files)[:3]:
        print('  '*(lvl+1) + fname)
    if len(files) > 3:
        print('  '*(lvl+1) + f'... ({len(files)} total)')

# ── 6. Auto-install missing deps ─────────────────────────────
import importlib
missing = [p for m,p in {'torchmetrics':'torchmetrics','lpips':'lpips',
    'einops':'einops','timm':'timm','plyfile':'plyfile'}.items()
           if importlib.util.find_spec(m) is None]
if missing:
    print(f'\nAuto-installing: {missing}')
    subprocess.run([sys.executable,'-m','pip','install','-q']+missing, check=True)

# ── 7. Launch training with only valid args ───────────────────
import torch
print(f'\n🚀 Launching training on {torch.cuda.get_device_name(0)}...')

# Read train.py --help to get the exact valid arg list
help_out = subprocess.run([sys.executable,'/content/EndoGaussian/train.py','--help'],
                          capture_output=True, text=True).stdout + \
           subprocess.run([sys.executable,'/content/EndoGaussian/train.py','--help'],
                          capture_output=True, text=True).stderr

def arg_exists(name):
    return f'--{name}' in help_out

train_cmd = [
    sys.executable, '/content/EndoGaussian/train.py',
    '-s', DATASET,
    '--model_path', GAUSSIANS_OUT,
    '--iterations', '20000',
    '--densify_until_iter', '15000',
    '--opacity_reset_interval', '3000',
    '--save_iterations', '7000', '14000', '20000',
    '--test_iterations', '7000', '14000', '20000',
    '--eval',
]
# Add optional args only if this build of train.py actually supports them
optional = [
    ('deform_type',           ['--deform_type', 'node']),
    ('node_num',              ['--node_num', '512']),
    ('hyper_dim',             ['--hyper_dim', '8']),
    ('is_6dof',               ['--is_6dof', '0']),
    ('use_mfs',               ['--use_mfs', '1']),
    ('mfs_weight',            ['--mfs_weight', '0.1']),
    ('coarse_iterations',     ['--coarse_iterations', '3000']),
    ('expname',               ['--expname', 'endoscope_4d']),
]
for argname, argval in optional:
    if arg_exists(argname):
        train_cmd += argval
        print(f'  + --{argname} {argval[1]}')
    else:
        print(f'  - skipping --{argname} (not in this build)')

print(f'\nFull command:')
print(' '.join(train_cmd[:10]), '...')
print()

proc = subprocess.Popen(train_cmd, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
log = []
for line in proc.stdout:
    print(line, end='', flush=True)
    log.append(line)
proc.wait()

if proc.returncode != 0:
    full = ''.join(log)
    print('\n❌ Training failed — last 40 lines:')
    print(''.join(log[-40:]))
    if 'CUDA out of memory' in full:
        print('💡 OOM: re-run with --node_num 256 or reduce --iterations to 10000')
    elif 'Could not recognize scene' in full:
        print('💡 Scene format still wrong — paste the dataset layout printed above')
    elif 'ModuleNotFoundError' in full:
        import re
        print(f"💡 Run: !pip install {' '.join(re.findall(r\"No module named '([^']+)'\", full))}")
    raise RuntimeError('Training failed — see output above')

# ── 8. Verify outputs ─────────────────────────────────────────
cfg_path  = f'{GAUSSIANS_OUT}/cfg_args'
ply_files = list(Path(GAUSSIANS_OUT).rglob('point_cloud.ply'))
print(f'\ncfg_args : {"✅" if os.path.exists(cfg_path) else "❌"}')
print(f'.ply files: {len(ply_files)}')
if not os.path.exists(cfg_path):
    import shlex
    Path(cfg_path).write_text(' '.join(shlex.quote(str(a)) for a in train_cmd))
    print(f'  wrote fallback cfg_args')

print('\n✅ Training complete!')


---
## 🎥 STEP 6 — Render & Export 4D Scene

In [ ]:
# ── Render novel-view and temporal playback ──────────────────
import os, subprocess, sys
from pathlib import Path

# ── Pre-flight checks before calling render.py ───────────────
cfg_path = f'{GAUSSIANS_OUT}/cfg_args'
ply_files = list(Path(GAUSSIANS_OUT).rglob('point_cloud.ply'))

print('Checking training outputs...')
print(f'  GAUSSIANS_OUT : {GAUSSIANS_OUT}')
print(f'  cfg_args      : {"✅" if os.path.exists(cfg_path) else "❌ missing"}')
print(f'  point_cloud   : {len(ply_files)} .ply file(s) found')

if not os.path.exists(cfg_path):
    print('\n❌ cfg_args not found — training output is incomplete.')
    print('   Most likely cause: training cell was not run, or crashed before saving.')
    print('   Fix: re-run the training cell (Step 5) and wait for it to finish.')
    raise FileNotFoundError(f'cfg_args missing at {cfg_path}')

if not ply_files:
    print('\n⚠️  No point_cloud.ply found — using latest available checkpoint...')
    # Find highest saved iteration
    iters = sorted([int(p.name) for p in Path(f'{GAUSSIANS_OUT}/point_cloud').iterdir()
                    if p.is_dir() and p.name.isdigit()], reverse=True) \
            if Path(f'{GAUSSIANS_OUT}/point_cloud').exists() else []
    render_iter = iters[0] if iters else 20000
    print(f'   Using iteration {render_iter}')
else:
    render_iter = 20000

# ── Run render.py ────────────────────────────────────────────
print(f'\nRendering 4D scene at iteration {render_iter}...')
render_cmd = [
    sys.executable, '/content/EndoGaussian/render.py',
    '-m', GAUSSIANS_OUT,
    '-s', DATASET,
    '--iteration', str(render_iter),
    '--skip_train',
]

result = subprocess.run(render_cmd, capture_output=True, text=True)
if result.returncode != 0:
    print('❌ render.py failed:')
    print(result.stdout[-2000:])
    print(result.stderr[-2000:])
    raise RuntimeError('Rendering failed')

print(result.stdout)
print('\n✅ Renders saved')


In [ ]:
# ── Run evaluation metrics ───────────────────────────────────
print("Computing PSNR / SSIM / LPIPS metrics...")
!python /content/EndoGaussian/metrics.py \
    -m {GAUSSIANS_OUT} \
    --iteration 20000

# Print results
results_path = f"{GAUSSIANS_OUT}/results.json"
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print("\n📊 Reconstruction Quality:")
    for k, v in results.items():
        print(f"   {k:10s}: {v:.4f}")

In [ ]:
# ── Compile renders into MP4 video ───────────────────────────
renders_dir = f"{GAUSSIANS_OUT}/test/ours_20000/renders"
depth_dir   = f"{GAUSSIANS_OUT}/test/ours_20000/depth"

output_video = f"{WORK_DIR}/renders/4D_reconstruction_Video01.mp4"

if os.path.exists(renders_dir) and len(os.listdir(renders_dir)) > 0:
    print("Compiling render frames into video...")
    !ffmpeg -framerate 15 \
        -i {renders_dir}/%05d.png \
        -c:v libx264 -pix_fmt yuv420p -crf 18 \
        {output_video} -y -loglevel error
    size_mb = os.path.getsize(output_video) / 1e6
    print(f"✅ Output video: {output_video} ({size_mb:.1f} MB)")
else:
    print("⚠️  Renders directory empty — check training logs above")

In [ ]:
# ── Visualize a sample reconstruction result ─────────────────
if os.path.exists(renders_dir):
    render_frames = sorted(Path(renders_dir).glob("*.png"))
    gt_frames     = sorted(Path(f"{DATASET}/images").glob("*.png"))

    n_show = min(3, len(render_frames), len(gt_frames))
    idxs = [0, len(render_frames)//2, len(render_frames)-1][:n_show]

    fig, axes = plt.subplots(2, n_show, figsize=(5*n_show, 8))
    fig.patch.set_facecolor('#111')

    for col, idx in enumerate(idxs):
        gt  = cv2.cvtColor(cv2.imread(str(gt_frames[idx])), cv2.COLOR_BGR2RGB)
        rnd = cv2.cvtColor(cv2.imread(str(render_frames[idx])), cv2.COLOR_BGR2RGB)
        axes[0, col].imshow(gt)
        axes[0, col].set_title(f"Ground Truth (t={idx/(len(render_frames)-1):.2f})",
                               color='#4ade80', fontsize=9)
        axes[0, col].axis('off')
        axes[1, col].imshow(rnd)
        axes[1, col].set_title("4D Gaussian Render", color='#60a5fa', fontsize=9)
        axes[1, col].axis('off')

    plt.suptitle("Ground Truth vs 4D Gaussian Reconstruction",
                 color='white', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## 💾 STEP 7 — Save All Outputs to Google Drive

In [ ]:
# ── Save everything to Google Drive ─────────────────────────
# Colab local storage is WIPED when the session ends.
# This cell copies everything important to your Drive.

import shutil
print(f"Saving outputs to Google Drive: {SAVE_DIR}/")

# 1. Trained model (gaussian .ply files)
if os.path.exists(GAUSSIANS_OUT):
    dst = f"{SAVE_DIR}/gaussians"
    if os.path.exists(dst): shutil.rmtree(dst)
    shutil.copytree(GAUSSIANS_OUT, dst)
    print(f"  ✅ Gaussian model → {dst}")

# 2. Output video
if os.path.exists(output_video):
    shutil.copy(output_video, f"{SAVE_DIR}/4D_reconstruction_Video01.mp4")
    print(f"  ✅ Render video  → {SAVE_DIR}/4D_reconstruction_Video01.mp4")

# 3. Depth maps
depth_dst = f"{SAVE_DIR}/depth_maps"
if os.path.exists(f"{WORK_DIR}/depth/vis"):
    if os.path.exists(depth_dst): shutil.rmtree(depth_dst)
    shutil.copytree(f"{WORK_DIR}/depth/vis", depth_dst)
    print(f"  ✅ Depth maps    → {depth_dst}")

# 4. Dataset (prepared frames + transforms)
dataset_dst = f"{SAVE_DIR}/dataset"
if os.path.exists(DATASET):
    if os.path.exists(dataset_dst): shutil.rmtree(dataset_dst)
    shutil.copytree(DATASET, dataset_dst)
    print(f"  ✅ Dataset       → {dataset_dst}")

# 5. Metrics
if os.path.exists(results_path):
    shutil.copy(results_path, f"{SAVE_DIR}/metrics.json")
    print(f"  ✅ Metrics       → {SAVE_DIR}/metrics.json")

print(f"\n🎉 All outputs saved to Google Drive → {SAVE_DIR}/")

In [ ]:
# ── Download the output video directly ──────────────────────
from google.colab import files

if os.path.exists(output_video):
    print("Preparing download...")
    files.download(output_video)
    print("✅ Download started")
else:
    print("⚠️  No output video found — re-run Step 6")

---
## 🛟 Troubleshooting

| Problem | Fix |
|---|---|
| `CUDA out of memory` | Runtime → Disconnect → Reconnect. If using T4, reduce `--node_num 256` |
| `COLMAP model empty` | Normal for laparoscopy. Pipeline falls back to depth-only init automatically |
| `Session disconnected` | All intermediate files are in Drive. Re-run from the step that failed |
| Renders look blurry | Increase `--iterations 30000` in Step 5 |
| Depth maps look wrong | The model is still valid — depth is only used for initialization |

### Runtime Tips
- Use **Colab Pro** (A100 GPU) to cut training time from 15 min → 3 min
- Enable **"Background execution"** in Colab Pro to prevent disconnection
- The `SAVE_DIR` on Drive is your checkpoint — re-run only failed cells after a crash

### Expected Metrics (from EndoGaussian paper)
- **PSNR**: ~38.5 dB (higher = better)
- **SSIM**: ~0.95 (higher = better)
- **LPIPS**: ~0.05 (lower = better)
- **Render speed**: ~168 FPS
